

# **0. Introduction and general notes about the dataset:**

### **0.1 Variables Details:**

1. **Date:** Specific date when the transaction was done.
2. **Supplier Name:** Name of supplier.
3. **CAT:** Cagtegory of the plastic being recycled.
4. **Price:** Price per kilogram of bales.
5. **Weight:** Weight of bales.
6. **Deduction:** Percentage of bales deducted from the total weight.
7. **Accept:** Accepted weight.
8. **Reject:** Rejected weight.
9. **Cost:** Cost of weight of the material being recycled.
10. **Innvoice:** The actual money being paid including shipping.
11. **Carta Series:** Code of Carta
12. **Supply Area:** Area where material was supplied.
13. **Bulking Station Name**: Bulking Station Name.


###**0.2 General Plan:**
1. Data collection (aggregation of 3 sheets (2021, 2022, 2023)
2. Data Cleaning
3. Exploratory data analysis
4. Feature Engineering
5. Model selection
6. Model Training
7. Model Deployment



###**0.3 Detailed Plan:**
**Time Series Analysis: Step-by-Step Plan**

---

#### **1. Define the Objective**  
- Determine the goal of the analysis (e.g., forecasting sales, detecting anomalies, identifying trends).  
- Understand the business or research context.

---

#### **2. Collect and Prepare Data**  
- **Gather Data**: Ensure the dataset contains time-stamped observations.  
- **Check Granularity**: Daily, weekly, monthly, or yearly data? Ensure consistency.  
- **Handle Missing Values**:
  - Forward fill, backward fill, interpolation, or imputation.
- **Handle Outliers**:
  - Use box plots, rolling averages, or winsorization to detect and manage anomalies.
- **Convert to DateTime Format**:
  - Ensure the time column is in a proper `datetime` format for analysis.

---

#### **3. Explore the Data (EDA - Exploratory Data Analysis)**  
- **Plot the Time Series**:
  - Use visualization tools to identify trends and patterns.
  
- **Check for Trends**:
  - Identify upward or downward movements over time.
  
- **Check for Seasonality**:
  - Look for repeating patterns (e.g., daily, weekly, yearly cycles).
  
- **Check for Stationarity**:
  - Use rolling mean and variance to check if statistical properties change over time.
  - Apply **Dickey-Fuller Test** to confirm stationarity.
  
- If `p-value < 0.05`, the data is stationary.

---

#### **4. Transform Data (If Needed)**  
- **Make Data Stationary**:
  - Differencing: Removing trends by subtracting previous values.
  - Log transformation: Reducing variance.
  - Seasonal decomposition: Breaking data into trend, seasonal, and residual components.

---

#### **5. Model Selection**  
- **Baseline Models**:
  - Naïve forecast: Uses the last known value.
  - Moving averages: Smooths fluctuations.

- **Statistical Models**:
  - **ARIMA (AutoRegressive Integrated Moving Average)**: Works well for stationary data.
  - **SARIMA (Seasonal ARIMA)**: Useful for seasonal data.

- **Machine Learning Models**:
  - Random Forest, XGBoost, LSTMs (for deep learning approaches).

- **Deep Learning Models**:
  - LSTMs, GRUs, Transformers for more complex time-series predictions.

---

#### **6. Model Evaluation**  
- Use **train-test split** (e.g., 80%-20% split).
- Common evaluation metrics:
  - **MAE (Mean Absolute Error)**
  - **MSE (Mean Squared Error)**
  - **RMSE (Root Mean Squared Error)**
  - **MAPE (Mean Absolute Percentage Error)**

---

#### **7. Forecasting Future Values**  
- Use the trained model to predict future values.
- Plot actual vs. predicted values for validation.
- Forecast multiple time steps ahead.

---

#### **8. Deploy & Monitor**  
- Automate forecasts using scripts or dashboards.
- Set up alerts for anomalies.
- Continuously refine the model as new data arrives.

---



### **0.4 Objective:**
Forecasting the price of Bales quarterly.


### **1.0 Data Loading and Initial Exploration**

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')
import os

color_pal = sns.color_palette()
plt.style.use('fivethirtyeight')

#Installing a new version of scikit-learn
!pip uninstall -y scikit-learn
!pip install scikit-learn==1.5.2

import xgboost as xgb
from sklearn.metrics import mean_squared_error

#Mounting colab to drive
from google.colab import drive
drive.mount('/content/drive')

#Uploading the file
from google.colab import files
uploaded = files.upload()



In [ ]:
#Creating the dataframe
import pandas as pd
PET_data = pd.read_csv('/content/PET Local Prices Sheet.csv')
exchage_rates = pd.read_csv('/content/Official Exchange Rates Historical.csv')


In [ ]:
data = pd.merge(PET_data, exchage_rates, on='Date', how='left')

In [ ]:
#Exploring the data frame:
data.head()

In [ ]:
data.shape

In [ ]:
data.info()

In [ ]:
data.columns

In [ ]:
data.describe().T

### **Insights and more understanding of the dataset:**
1. the price range has increased exponentially over the 3 years.
2. the weight has a mean of 7000 while the max is 32000 which means that there are some outliers that skew the data.


### **1.1 Cleaning the Dataset:**
1. Checking the data types of each column
2. Setting the index of the data to date.
3. Replacing wrong format data.
4. Removing Duplicates & Nulls.
5. Dealing with outliers.
6. Adding Number of Shipments column


In [ ]:
#Checking for Duplicates:
data.duplicated().sum()

In [ ]:
#Checking for Duplicates:
data.drop_duplicates(inplace=True)
data.duplicated().sum()
#No more duplicates!

In [ ]:
#Checking for Nulls:
data.isnull().sum()


In [ ]:
#filling null values in price column using forward fill method:
data['Price'].fillna(method = 'ffill', inplace=True)
data['Sell'].fillna(method = 'bfill', inplace=True)

data.isnull().sum()

In [ ]:
data['Sell'].fillna(method = 'ffill', inplace=True)
data.isnull().sum()
#No more null values!

In [ ]:
#Creating the new column: Getting the total number of transactions each day:
data['Number_of_Shipments'] = 1
data.head()

In [ ]:
#Scaling the Price column to account for inflation:
data['Unit_Price_in_USD'] =  data['Price'] / data['Sell']
data.head()

In [ ]:
#Setting the Date column to datetime format
data['Date'] = pd.to_datetime(data['Date'])
data.set_index('Date', inplace = True)
data.head()

In [ ]:
#Dealing with the date column and aggregating the weight in each day
data = data.resample('D').agg({'Price':'mean', 'Sell':'mean', 'Weight':'sum', 'Number_of_Shipments': 'sum', 'Unit_Price_in_USD': 'mean'})
data.head()

In [ ]:
#Recheck for nulls:
data.isnull().sum()

In [ ]:
data['Price'] = data['Price'].fillna(method = 'ffill')
data['Sell'] = data['Sell'].fillna(method = 'ffill')
data['Unit_Price_in_USD'] = data['Unit_Price_in_USD'].fillna(method = 'ffill')
data.isnull().sum()

#No Null values!

In [ ]:
#Rechecking Duplicates
data.duplicated().sum()

In [ ]:
data = data.drop_duplicates()
data.duplicated().sum()
#No more duplicates!

In [ ]:
data.reset_index(inplace=True)
#Adding new day, month, year, day of year, quarter columns
data['Day'] = data['Date'].dt.day
data['Month'] = data['Date'].dt.month
data['Year'] = data['Date'].dt.year
data['Day_of_the_year'] = data['Date'].dt.dayofyear
data['Quarter'] = data['Date'].dt.quarter
data['week_of_year'] = data['Date'].dt.isocalendar().week

data['Date'] = pd.to_datetime(data['Date'])
data.set_index('Date', inplace = True)
data.head()

In [ ]:
data.columns

In [ ]:
#Removing Unnecessary Columns
data = data[['Unit_Price_in_USD', 'Day', 'Month', 'Year', 'Day_of_the_year', 'Quarter', 'week_of_year']]
data.head()

### **1.2 Outlier Detection and Removal:**
1. Visualize individual variables: Since data is skewed, we will be using the boxplot.
2. Use IQR thresholds to detect outliers, MAD (Mean Absolute Deviation).
3. If the above methods fail to detect the outliers since the data seems not to exhibit a specific pattern, we will be using the ML model (Isolation Forest, or Local Outlier Factor (LOF)) depending on the case.
4. Re-visualize to see the data without outliers.

In [ ]:
#Aggregated data by date have less observations since the three years have in total 1095 before removing duplicates and null values!
data.shape

In [ ]:
#Visualising key variables:

#Detecting outliers in the Unit Price variable:
plt.figure(figsize = (10,6))
sns.histplot(x= data['Unit_Price_in_USD'], kde= True, color = 'teal', bins = 30)
plt.title('Distribution of Unit Price in USD')
plt.xlabel('Unit Price in USD')
plt.ylabel('Frequency')
plt.show()

#The Unit price is not normal so we cannot use the z-scores.
#we will check with the boxplot and then we use the IQR method to detect specific observations.
#It does not seem that we have outliers in the Unit Price column.



In [ ]:
#Unit Price in USD Boxplot:
plt.figure(figsize = (10,6))
sns.boxplot(x= data['Unit_Price_in_USD'], color = 'DeepPink')
plt.title('Distribution of Unit Price in USD')
plt.xlabel('Unit Price in USD')
plt.ylabel('Frequency')
plt.show()

#No outliers Detected in the Price column!!

### **1.3 Visualizing Key Variables to spot trends:**
1. Price (Unit price in USD) per kg over the three years: spot the trend.
2. Average Price by Month
3. Seasonal Price fluctuation
4. Average Price change in a Month/Quarter.


In [ ]:
#Plotting the Price change over time
plt.figure(figsize = (10,6))

plt.plot(data.index,
         data['Unit_Price_in_USD'],
         label = 'Unit Price in USD',
         color = 'orange')

plt.title('Unit Price in USD Over Time')
plt.xlabel('Date')
plt.ylabel('Unit Price in USD')
plt.legend()
plt.show()


In [ ]:
#Checking for Monthly changes - 2021:
plt.figure(figsize = (10, 6))
one_yr_data = data.loc['2021-01-01':'2021-12-31']
sns.lineplot(x = one_yr_data.index.month , y = 'Unit_Price_in_USD', data = one_yr_data, color = 'orangered')



#Checking for Monthly changes - 2022:
one_yr_data = data.loc['2022-01-01':'2022-12-31']
sns.lineplot(x = one_yr_data.index.month , y = 'Unit_Price_in_USD', data = one_yr_data, color = 'lawngreen')



#Checking for Monthly changes -2023:
one_yr_data = data.loc['2023-01-01':'2023-12-31']
sns.lineplot(x = one_yr_data.index.month , y = 'Unit_Price_in_USD', data = one_yr_data, color = 'brown')


plt.title('Monthly Unit Price in USD for each year')
plt.xlabel('Month')
plt.ylabel('Unit Price in USD')

plt.legend(['2023', '2022', '2021'], fontsize = 12, title = "Years")
plt.grid(linestyle = '--', alpha = 0.7)
plt.show()


In [ ]:
#Seasonal Change of Price
plt.figure(figsize = (10,6))
sns.boxplot(x ='Quarter', y = 'Unit_Price_in_USD', data = data.reset_index() , color = 'magenta')
plt.title('Seasonal Change of Unit Price in USD by Quarter')
plt.xlabel('Quarter')
plt.ylabel('Unit Price in USD')
plt.grid(linestyle = '--', alpha = 0.7)

plt.show()


In [ ]:

#Monthly Change of Price
plt.figure(figsize = (10,6))
sns.boxplot(x ='Month', y = 'Unit_Price_in_USD', data = data.reset_index() , color = 'crimson')
plt.title('Seasonal Change of Unit Price in USD by Month')
plt.xlabel('Month')
plt.ylabel('Unit Price in USD')
plt.show()


In [ ]:
#Average Price by month
avg_price_by_month = data.groupby('Month')['Unit_Price_in_USD'].mean()
plt.figure(figsize = (10,6))
sns.barplot(x = avg_price_by_month.index, y = avg_price_by_month.values, color = 'deeppink')
plt.title('Average Unit Price by Month')
plt.xlabel('Month')
plt.ylabel('Average Unit Price in USD')
plt.show()



In [ ]:
#Plotting weeks of Price (Boxplots)

plt.figure(figsize = (20,10))
sns.boxplot(x = 'week_of_year', y = 'Unit_Price_in_USD', data = data, color = color_pal[1])
plt.title('Change in Price for different weeks in a year')
plt.xlabel('Week of Year')
plt.ylabel('Unit Price in USD')

### **2.0 Machine Learning:**


In [ ]:
#Train/ Test Split
train = data.loc[data.index <'2022-12-31']
validation = data.loc[(data.index >= '2022-12-31') & (data.index < '2023-06-01')]
test = data.loc[data.index >= '2023-06-01']

fig, ax = plt.subplots(figsize = (10,6))
train['Unit_Price_in_USD'].plot(ax = ax, label = 'Training Set')
validation['Unit_Price_in_USD'].plot(ax = ax, label = 'Validation Set')
test['Unit_Price_in_USD'].plot(ax = ax, label = 'Test Set')

ax.legend(['Training Set', 'Validation Set', 'Test Set'])
plt.axvline('2023-01-01', color = 'black', ls = '--')
plt.axvline('2023-06-01', color = 'black', ls = '--')
plt.title('Train/Validation/Test Data Split')
plt.show()


In [ ]:
#plotting one week of data (Jan 2021, wk 1)
data.loc[(data.index > '2022-07-31') & (data.index < '2022-08-08')]['Unit_Price_in_USD'].plot(figsize = (10,6),
                                                                                              color = color_pal[0],
                                                                                              title = 'Price in the First Week in August 2022')

plt.show()


In [ ]:
#plotting one month of data
data.loc[(data.index > '2021-12-31') & (data.index < '2022-02-01')]['Unit_Price_in_USD'].plot(figsize = (10,6),
                                                                                              color = color_pal[1],
                                                                                              title = 'Price fluctuation in January 2022')
plt.show()


In [ ]:
data.columns

In [ ]:
Features = ['Day', 'Month', 'Year', 'Day_of_the_year',
       'Quarter', 'week_of_year']

Target = 'Unit_Price_in_USD'

#Training Set
x_train = train[Features]
y_train = train[Target]

#Validation Set
x_validation = validation[Features]
y_validation = validation[Target]

#Test Set (not fed into the model)
x_test = test[Features]
y_test = test[Target]

In [ ]:
Reg_model = xgb.XGBRegressor(n_estimators = 100,
                             early_stopping_rounds = 50,
                             learning_rate = 0.000001)

Reg_model.fit(x_train,
              y_train,
              eval_set = [(x_train, y_train), (x_validation, y_validation)],
               verbose = True)

In [ ]:
test['Prediction'] = Reg_model.predict(x_test)
test.head()

In [ ]:
data2 = pd.merge(data, test['Prediction'], how = 'left', left_index = True, right_index = True)


In [ ]:
data2.tail(500)

In [ ]:
ax = data[['Unit_Price_in_USD']].plot(figsize = (15,5))
data2['Prediction'].plot(ax= ax, style = '.')
plt.title('Actual vs. Predicted Price')
plt.xlabel('Date')
plt.ylabel('Unit Price in USD')
plt.legend(['Actual', 'Predicted'])
plt.axvline('2023-06-01', color = 'black', ls = '--')
plt.show()
